In [13]:
import os
from groq import Groq

client = Groq(api_key=os.environ.get("GROQ_API_KEY"))
print("Groq Connected")


In [14]:
import json

with open(
    "../data/processed_papers/chunked_papers.json",
    "r",
    encoding="utf-8"
) as f:
    chunks = json.load(f)

print(len(chunks))

643


In [17]:
def extract_factors(text):

    prompt = f"""
Extract:

1. Risk factors
2. Alpha factors
3. Strategy type
4. Performance metrics
5. Key findings

Text:

{text}
"""

    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[
            {"role": "user", "content": prompt}
        ]
    )

    return response.choices[0].message.content

In [18]:
sample = chunks[0]["text"]

result = extract_factors(sample)

print(result)

Based on the provided text, the following information can be extracted:

1. **Risk factors**: Not mentioned in the text.
2. **Alpha factors**: Not mentioned in the text.
3. **Strategy type**: Not mentioned in the text, but the paper appears to focus on a mathematical approach to option pricing under non-Markovian stochastic volatility models.
4. **Performance metrics**: Not mentioned in the text.
5. **Key findings**: The paper proposes a deep signature approach for representing rough stochastic differential equations, which allows for the application of standard analytical tools to option pricing problems under non-Markovian stochastic volatility models.

Note: The text appears to be an abstract for a research paper and does not provide detailed information on risk factors, alpha factors, strategy type, or performance metrics. The key findings mentioned are based on the brief description of the paper's approach and methodology.


In [19]:
import pandas as pd
import time

factor_database = []

MAX_CHUNKS = 10

for i, chunk in enumerate(chunks[:MAX_CHUNKS]):

    print(f"\nProcessing {i+1}/{MAX_CHUNKS}")

    success = False

    for attempt in range(3):

        try:

            extracted = extract_factors(
                chunk["text"][:2000]
            )

            factor_database.append({
                "paper_title": chunk["paper_title"],
                "factors": extracted
            })

            print("✓ Done:",
                  chunk["paper_title"])

            success = True

            break

        except Exception as e:

            print(
                f"Retry {attempt+1}/3:",
                str(e)[:100]
            )

            time.sleep(10)

    if not success:

        print(
            "✗ Skipped:",
            chunk["paper_title"]
        )

    time.sleep(3)

print("\nFinished!")
print("Successful extractions:",
      len(factor_database))


Processing 1/10
✓ Done: Option pricing under non-Markovian stochastic volatility models: A deep

Processing 2/10
✓ Done: Option pricing under non-Markovian stochastic volatility models: A deep

Processing 3/10
✓ Done: Option pricing under non-Markovian stochastic volatility models: A deep

Processing 4/10
✓ Done: Option pricing under non-Markovian stochastic volatility models: A deep

Processing 5/10
✓ Done: Option pricing under non-Markovian stochastic volatility models: A deep

Processing 6/10
✓ Done: Option pricing under non-Markovian stochastic volatility models: A deep

Processing 7/10
✓ Done: Option pricing under non-Markovian stochastic volatility models: A deep

Processing 8/10
✓ Done: Option pricing under non-Markovian stochastic volatility models: A deep

Processing 9/10
✓ Done: Option pricing under non-Markovian stochastic volatility models: A deep

Processing 10/10
✓ Done: Option pricing under non-Markovian stochastic volatility models: A deep

Finished!
Successful extract

In [20]:
df = pd.DataFrame(
    factor_database
)

df.head()

,paper_title,factors
0,Option pricing under non-Markovian stochastic ...,"Unfortunately, the text does not provide infor..."
1,Option pricing under non-Markovian stochastic ...,"Based on the provided text, here are the extra..."
2,Option pricing under non-Markovian stochastic ...,"Based on the provided text, it appears that th..."
3,Option pricing under non-Markovian stochastic ...,"Based on the provided text, here are the extra..."
4,Option pricing under non-Markovian stochastic ...,"Based on the provided text, the extraction of ..."


In [21]:
df.to_csv(
    "../data/factor_database.csv",
    index=False
)

print("Factor database saved!")
print(df.shape)

Factor database saved!
(10, 2)


In [22]:
response = client.chat.completions.create(
    model="llama-3.3-70b-versatile",
    messages=[
        {"role": "user", "content": "Say hello"}
    ]
)

print(response.choices[0].message.content)

Hello. How can I assist you today?
